# Probability, Distributions, and Hypothesis Testing (Step-by-Step)

This notebook is designed for a data science student.

You will learn by:
1. Building intuition with simulations.
2. Visualizing distributions.
3. Running real hypothesis tests.
4. Interpreting results in plain language.


## Learning Goals

By the end, you should be able to:
- Explain probability in practical terms.
- Recognize common distributions and when to use them.
- Formulate null and alternative hypotheses.
- Run one-sample, two-sample, and categorical tests.
- Interpret p-values, confidence intervals, and effect sizes.


## 0) Setup


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Reproducibility
rng = np.random.default_rng(42)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)

print("Libraries imported successfully!")


Libraries imported successfully!


## 1) Probability Basics with Simulation

Probability is long-run frequency.
If you repeat an experiment many times, the observed proportion converges to the true probability.


In [ ]:
# Simulate coin flips and track running probability of heads
n_flips = 5000
flips = rng.choice([0, 1], size=n_flips, p=[0.5, 0.5])  # 1 = heads
running_p_heads = np.cumsum(flips) / np.arange(1, n_flips + 1)

plt.plot(running_p_heads, color="teal")
plt.axhline(0.5, color="red", linestyle="--", label="True probability = 0.5")
plt.title("Law of Large Numbers: Running Probability of Heads")
plt.xlabel("Number of flips")
plt.ylabel("Estimated P(Heads)")
plt.legend()
plt.show()

print(f"Final estimated P(Heads) after {n_flips} flips: {running_p_heads[-1]:.4f}")


### Quick Reflection
- Early estimates fluctuate a lot.
- With more trials, estimates stabilize near the true value.


## 2) Random Variables and Common Distributions

We use distributions to model uncertainty.

### 2.1 Bernoulli Distribution
A single trial with two outcomes (0/1), like conversion or no conversion.


In [ ]:
p = 0.3
n = 10000
bernoulli_samples = rng.binomial(n=1, p=p, size=n)

counts = pd.Series(bernoulli_samples).value_counts().sort_index()
proportions = counts / n

ax = proportions.plot(kind="bar", color=["slateblue", "orange"])
ax.set_xticklabels(["0 (Failure)", "1 (Success)"], rotation=0)
plt.title("Bernoulli Distribution (Simulated)")
plt.ylabel("Proportion")
plt.show()

print("Estimated success probability:", proportions.loc[1])


### 2.2 Binomial Distribution
Counts successes in a fixed number of Bernoulli trials.
Example: number of sign-ups out of 20 visitors.


In [ ]:
n_trials = 20
p_success = 0.4
n_experiments = 10000
binom_samples = rng.binomial(n=n_trials, p=p_success, size=n_experiments)

sns.histplot(binom_samples, bins=np.arange(-0.5, n_trials + 1.5, 1), stat="probability", color="steelblue")
plt.title("Binomial Distribution: Successes out of 20")
plt.xlabel("Number of successes")
plt.ylabel("Probability")
plt.show()

print("Sample mean:", np.mean(binom_samples))
print("Theoretical mean n*p:", n_trials * p_success)


### 2.3 Normal Distribution
Common for continuous measurements (height, test scores, errors).


In [ ]:
mu, sigma = 100, 15
normal_samples = rng.normal(mu, sigma, 5000)

sns.histplot(normal_samples, kde=True, stat="density", color="seagreen")
plt.title("Normal Distribution (Simulated)")
plt.xlabel("Value")
plt.ylabel("Density")
plt.show()

print(f"Sample mean: {normal_samples.mean():.2f}")
print(f"Sample std:  {normal_samples.std(ddof=1):.2f}")


### 2.4 Right-Skewed Example (Exponential)
Useful for waiting times and time-to-event data.


In [ ]:
exp_samples = rng.exponential(scale=2.0, size=5000)

sns.histplot(exp_samples, kde=True, stat="density", color="darkorange")
plt.title("Exponential Distribution (Right-Skewed)")
plt.xlabel("Value")
plt.ylabel("Density")
plt.xlim(0, 15)
plt.show()


## 3) Sampling Distributions and the Central Limit Theorem (CLT)

Even if raw data is skewed, the **distribution of sample means** becomes approximately normal as sample size grows.


In [ ]:
# Population is skewed (exponential)
population = rng.exponential(scale=2.0, size=200000)

sample_sizes = [5, 30, 100]
means_dict = {}

for n in sample_sizes:
    means = [rng.choice(population, size=n, replace=False).mean() for _ in range(2000)]
    means_dict[n] = np.array(means)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, n in zip(axes, sample_sizes):
    sns.histplot(means_dict[n], bins=30, kde=True, stat="density", color="mediumpurple", ax=ax)
    ax.set_title(f"Sample mean distribution (n={n})")
    ax.set_xlabel("Sample mean")

axes[0].set_ylabel("Density")
plt.suptitle("Central Limit Theorem in Action", y=1.05)
plt.tight_layout()
plt.show()


## 4) Hypothesis Testing Framework

Every hypothesis test follows this logic:
1. Define parameter and claim.
2. State hypotheses:
   - Null hypothesis \(H_0\): no effect / baseline.
   - Alternative hypothesis \(H_1\): effect exists.
3. Choose significance level \(lpha\), commonly 0.05.
4. Compute test statistic and p-value.
5. Decide:
   - If p-value < \(lpha\): reject \(H_0\).
   - Otherwise: fail to reject \(H_0\).

Important: "fail to reject" is not "prove true."


## 5) One-Sample t-Test (Mean vs benchmark)

Scenario: A class claims average score is 70.
You sampled student scores and want to test this claim.


In [ ]:
scores = rng.normal(loc=74, scale=10, size=40)  # sample data
benchmark = 70
alpha = 0.05

# Two-sided one-sample t-test
t_stat, p_value = stats.ttest_1samp(scores, popmean=benchmark)

sample_mean = scores.mean()
sample_std = scores.std(ddof=1)
n = len(scores)
se = sample_std / np.sqrt(n)

# 95% confidence interval for mean
ci_low, ci_high = stats.t.interval(0.95, df=n-1, loc=sample_mean, scale=se)

print(f"n = {n}")
print(f"Sample mean = {sample_mean:.2f}")
print(f"t-statistic = {t_stat:.3f}")
print(f"p-value = {p_value:.5f}")
print(f"95% CI for mean = ({ci_low:.2f}, {ci_high:.2f})")
print("Decision:", "Reject H0" if p_value < alpha else "Fail to reject H0")


In [ ]:
# Visualize sample distribution with benchmark
sns.histplot(scores, kde=True, color="cornflowerblue")
plt.axvline(benchmark, color="red", linestyle="--", label="H0 mean = 70")
plt.axvline(scores.mean(), color="black", linestyle="-", label=f"Sample mean = {scores.mean():.2f}")
plt.title("One-Sample t-Test: Score Distribution")
plt.xlabel("Score")
plt.ylabel("Count")
plt.legend()
plt.show()


### Effect Size (Cohen's d for one sample)
Effect size tells us practical magnitude, not just statistical significance.


In [ ]:
cohens_d_one_sample = (sample_mean - benchmark) / sample_std
print(f"Cohen's d (one-sample): {cohens_d_one_sample:.3f}")


## 6) Two-Sample t-Test (A/B style comparison)

Scenario: Did a new study method improve scores vs old method?


In [ ]:
old_method = rng.normal(loc=68, scale=9, size=45)
new_method = rng.normal(loc=73, scale=9, size=45)
alpha = 0.05

# Welch's t-test (does not assume equal variances)
t_stat, p_value = stats.ttest_ind(new_method, old_method, equal_var=False)

mean_old = old_method.mean()
mean_new = new_method.mean()

# Cohen's d (pooled SD approximation)
pooled_sd = np.sqrt(((len(old_method)-1)*old_method.var(ddof=1) + (len(new_method)-1)*new_method.var(ddof=1)) / (len(old_method)+len(new_method)-2))
cohens_d = (mean_new - mean_old) / pooled_sd

print(f"Old mean = {mean_old:.2f}")
print(f"New mean = {mean_new:.2f}")
print(f"Difference = {mean_new - mean_old:.2f}")
print(f"t-statistic = {t_stat:.3f}")
print(f"p-value = {p_value:.6f}")
print(f"Cohen's d = {cohens_d:.3f}")
print("Decision:", "Reject H0" if p_value < alpha else "Fail to reject H0")


In [ ]:
df_plot = pd.DataFrame({
    "score": np.concatenate([old_method, new_method]),
    "group": ["Old"] * len(old_method) + ["New"] * len(new_method)
})

sns.boxplot(data=df_plot, x="group", y="score", palette=["lightcoral", "lightgreen"])
sns.stripplot(data=df_plot, x="group", y="score", color="black", alpha=0.5, size=4)
plt.title("Two-Sample Comparison: Old vs New Method")
plt.show()


## 7) Proportion Test (Conversion Rate)

Scenario: Baseline conversion is 12%. New landing page had 180 conversions out of 1200 visitors.
Test whether conversion changed.


In [ ]:
baseline_p = 0.12
conversions = 180
visitors = 1200

phat = conversions / visitors
se0 = np.sqrt(baseline_p * (1 - baseline_p) / visitors)
z_stat = (phat - baseline_p) / se0
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))  # two-sided

# 95% CI for observed proportion
se_hat = np.sqrt(phat * (1 - phat) / visitors)
ci_low = phat - 1.96 * se_hat
ci_high = phat + 1.96 * se_hat

print(f"Observed conversion rate = {phat:.4f}")
print(f"z-statistic = {z_stat:.3f}")
print(f"p-value = {p_value:.6f}")
print(f"95% CI for true conversion = ({ci_low:.4f}, {ci_high:.4f})")


In [ ]:
labels = ["Baseline", "Observed"]
values = [baseline_p, phat]

plt.bar(labels, values, color=["gray", "teal"])
plt.ylim(0, max(values) * 1.3)
plt.title("Baseline vs Observed Conversion")
plt.ylabel("Proportion")
for i, v in enumerate(values):
    plt.text(i, v + 0.005, f"{v:.3f}", ha="center")
plt.show()


## 8) Chi-Square Test of Independence (Categorical vs Categorical)

Scenario: Is pass/fail independent of teaching mode (online vs in-person)?


In [ ]:
contingency = np.array([
    [120, 30],   # online: pass, fail
    [150, 20]    # in-person: pass, fail
])

chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

print("Contingency table (observed):")
print(pd.DataFrame(contingency, index=["Online", "In-person"], columns=["Pass", "Fail"]))
print("\nExpected counts if independent:")
print(pd.DataFrame(expected, index=["Online", "In-person"], columns=["Pass", "Fail"]).round(2))
print(f"\nChi-square = {chi2:.3f}, dof = {dof}, p-value = {p_value:.6f}")
print("Decision:", "Reject independence" if p_value < 0.05 else "Fail to reject independence")


In [ ]:
obs_df = pd.DataFrame(contingency, index=["Online", "In-person"], columns=["Pass", "Fail"])
exp_df = pd.DataFrame(expected, index=["Online", "In-person"], columns=["Pass", "Fail"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(obs_df, annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title("Observed Counts")
sns.heatmap(exp_df, annot=True, fmt=".1f", cmap="Greens", ax=axes[1])
axes[1].set_title("Expected Counts (H0)")
plt.tight_layout()
plt.show()


## 9) Common Mistakes to Avoid

- Interpreting p-value as the probability that \(H_0\) is true.
- Saying "significant" without reporting effect size.
- Running many tests without multiple-testing correction.
- Ignoring assumptions (normality, independence, sample size, equal variance when required).
- Confusing statistical significance with practical significance.


## 10) Mini Practice Tasks

Try these on your own:
1. Change sample size in the one-sample t-test and observe p-value behavior.
2. Make the old/new means closer in the two-sample test and see when significance disappears.
3. Simulate non-normal data and compare t-test robustness with larger sample sizes.
4. Build your own chi-square table from a real classroom dataset.


## 11) Cheat Sheet (When to use which test)

- Mean vs known value: one-sample t-test.
- Mean vs mean (two groups): two-sample t-test (Welch by default).
- Proportion vs known value: one-proportion z-test.
- Association between categorical variables: chi-square test of independence.
